# Notebook 06: Customer Retention & Survival Analysis
**Focus:** Empirical shopping gap quantiles, defensible churn threshold, and cohort retention dynamics.


In [ ]:
import duckdb
import pandas as pd
import numpy as np
con = duckdb.connect('../data/analytical/retail_bi.duckdb', read_only=True)


### 1. Empirical Gap Distribution & Churn Threshold


In [ ]:
gaps = con.execute('''
    SELECT day - LAG(day) OVER (PARTITION BY household_key ORDER BY day) as gap
    FROM (SELECT DISTINCT household_key, day FROM fact_transactions)
''').df()['gap'].dropna()
print('Mean Gap:', gaps.mean(), 'Median:', gaps.median(), 'p90:', gaps.quantile(0.9), 'p95:', gaps.quantile(0.95))


### 2. Lapsed Rates by Customer Segment


In [ ]:
con.execute('''
    SELECT s.cluster_persona, COUNT(*) as total, SUM(r.is_lapsed) as lapsed,
           ROUND(SUM(r.is_lapsed)*100.0/COUNT(*), 2) as lapse_rate_pct
    FROM customer_segments s
    JOIN customer_retention_status r ON s.household_key = r.household_key
    GROUP BY s.cluster_persona ORDER BY lapse_rate_pct DESC
''').df()
